In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms, models
import pandas as pd
import os
from PIL import Image
import numpy as np
import matplotlib.pyplot as plt
from tqdm import tqdm
import torch.nn.functional as F
from torch.utils.data.distributed import DistributedSampler
from torch.nn.parallel import DistributedDataParallel as DDP
from skimage.transform import resize

In [ ]:
# --- CONFIGURATION ---
BATCH_SIZE = 128
IMAGE_SIZE = 224
DEVICE = torch.device("cuda:1" if torch.cuda.is_available() else "cpu")
CSV_PATH = r"/mnt/c/KLTN/LDM/data/list_attr_celeba.csv"
IMAGE_PATH = r"/mnt/c/KLTN/LDM/data/img_align_celeba/img_align_celeba"
PATH_CHECKPOINT = r"/mnt/e/KLTN/GAN/classifier/checkpoints/mobilenetv3_classifier_bce_non"
LEARNING_RATE = 0.001
NUM_EPOCHS = 30

# Danh sách attribute user yêu cầu
# Lưu ý: Cần map tên user gọi sang tên cột trong CSV CelebA
ATTR_MAP = {
    'Smiling': 'Smiling',
    'Eyeglasses': 'Eyeglasses',
    'Male': 'Gender (Male)',        # CelebA dùng 'Male'
    'Mouth_Slightly_Open': 'Open Mouth', # CelebA thường dùng cái này cho Open Mouth
    'No_Beard': 'No Beard',
    'Young': 'Age (Young)'          # CelebA dùng 'Young'
}
TARGET_ATTRS = list(ATTR_MAP.keys())
NUM_CLASSES = len(TARGET_ATTRS)

os.makedirs(PATH_CHECKPOINT, exist_ok=True)

In [3]:
# --- 1. LOSS FUNCTION: ASYMMETRIC LOSS ---
class AsymmetricLossOptimized(nn.Module):
    def __init__(self, gamma_neg=4, gamma_pos=1, clip=0.05, eps=1e-8, disable_torch_grad_focal_loss=False):
        super(AsymmetricLossOptimized, self).__init__()
        self.gamma_neg = gamma_neg
        self.gamma_pos = gamma_pos
        self.clip = clip
        self.disable_torch_grad_focal_loss = disable_torch_grad_focal_loss
        self.eps = eps

    def forward(self, x, y):
        # x: input logits, y: targets (multi-label binarized vector)
        
        # Calculating Probabilities
        x_sigmoid = torch.sigmoid(x)
        xs_pos = x_sigmoid
        xs_neg = 1 - x_sigmoid

        # Asymmetric Clipping
        if self.clip is not None and self.clip > 0:
            xs_neg = (xs_neg + self.clip).clamp(max=1)

        # Basic CE calculation
        los_pos = y * torch.log(xs_pos.clamp(min=self.eps))
        los_neg = (1 - y) * torch.log(xs_neg.clamp(min=self.eps))
        loss = los_pos + los_neg

        # Asymmetric Focusing
        if self.gamma_neg > 0 or self.gamma_pos > 0:
            if self.disable_torch_grad_focal_loss:
                torch.set_grad_enabled(False)
            pt0 = xs_pos * y
            pt1 = xs_neg * (1 - y)  # pt = p if t > 0 else 1-p
            pt = pt0 + pt1
            one_sided_gamma = self.gamma_pos * y + self.gamma_neg * (1 - y)
            one_sided_w = torch.pow(1 - pt, one_sided_gamma)
            if self.disable_torch_grad_focal_loss:
                torch.set_grad_enabled(True)
            loss *= one_sided_w

        return -loss.sum() / x.size(0) # Average over batch

class ChannelAttention(nn.Module):
    def __init__(self, in_planes, ratio=16):
        super(ChannelAttention, self).__init__()
        self.avg_pool = nn.AdaptiveAvgPool2d(1)
        self.max_pool = nn.AdaptiveMaxPool2d(1)
        self.fc1 = nn.Conv2d(in_planes, in_planes // ratio, 1, bias=False)
        self.relu = nn.ReLU()
        self.fc2 = nn.Conv2d(in_planes // ratio, in_planes, 1, bias=False)
        self.sigmoid = nn.Sigmoid()

    def forward(self, x):
        avg_out = self.fc2(self.relu(self.fc1(self.avg_pool(x))))
        max_out = self.fc2(self.relu(self.fc1(self.max_pool(x))))
        out = avg_out + max_out
        return self.sigmoid(out)

class SpatialAttention(nn.Module):
    def __init__(self, kernel_size=7):
        super(SpatialAttention, self).__init__()
        assert kernel_size in (3, 7), 'kernel size must be 3 or 7'
        padding = 3 if kernel_size == 7 else 1
        self.conv1 = nn.Conv2d(2, 1, kernel_size, padding=padding, bias=False)
        self.sigmoid = nn.Sigmoid()

    def forward(self, x):
        avg_out = torch.mean(x, dim=1, keepdim=True)
        max_out, _ = torch.max(x, dim=1, keepdim=True)
        x = torch.cat([avg_out, max_out], dim=1)
        x = self.conv1(x)
        return self.sigmoid(x)

class CBAMBlock(nn.Module):
    def __init__(self, in_planes, ratio=16, kernel_size=7):
        super(CBAMBlock, self).__init__()
        self.ca = ChannelAttention(in_planes, ratio)
        self.sa = SpatialAttention(kernel_size)

    def forward(self, x):
        out = x * self.ca(x)
        out = out * self.sa(out)
        return out

class ResNet18_CBAM(nn.Module):
    def __init__(self, num_classes):
        super(ResNet18_CBAM, self).__init__()
        # Load pre-trained ResNet18
        base_model = models.resnet18(weights=models.ResNet18_Weights.IMAGENET1K_V1)
        
        # Tách các layer để chèn CBAM vào giữa các stage
        self.stem = nn.Sequential(
            base_model.conv1, base_model.bn1, base_model.relu, base_model.maxpool
        )
        self.layer1 = base_model.layer1
        self.cbam1 = CBAMBlock(64) # ResNet18 layer1 out channels = 64
        
        self.layer2 = base_model.layer2
        self.cbam2 = CBAMBlock(128) # ResNet18 layer2 out channels = 128
        
        self.layer3 = base_model.layer3
        self.cbam3 = CBAMBlock(256) # ResNet18 layer3 out channels = 256
        
        self.layer4 = base_model.layer4
        self.cbam4 = CBAMBlock(512) # ResNet18 layer4 out channels = 512
        
        self.avgpool = base_model.avgpool
        self.fc = nn.Linear(512, num_classes)
    def forward(self, x):
        x = self.stem(x)
        
        x = self.layer1(x)
        x = self.cbam1(x) # Apply CBAM
        
        x = self.layer2(x)
        x = self.cbam2(x) # Apply CBAM
        
        x = self.layer3(x)
        x = self.cbam3(x) # Apply CBAM
        
        x = self.layer4(x)
        x = self.cbam4(x) # Apply CBAM
        
        x = self.avgpool(x)
        x = torch.flatten(x, 1)
        x = self.fc(x)
        return x

# --- 4. DATASET ---
class CelebADataset(Dataset):
    def __init__(self, csv_path, img_dir, attributes, transform=None):
        self.img_dir = img_dir
        self.transform = transform
        self.attributes = attributes
        
        # Đọc CSV
        df = pd.read_csv(csv_path)
        # Lấy image_id và các attribute cần thiết
        self.image_ids = df['image_id'].values
        self.labels = df[self.attributes].values
        
        # Xử lý label -1 thành 0
        self.labels[self.labels == -1] = 0
        self.labels = self.labels.astype(np.float32)

    def __len__(self):
        return len(self.image_ids)

    def __getitem__(self, idx):
        img_name = self.image_ids[idx]
        img_path = os.path.join(self.img_dir, img_name)
        
        image = Image.open(img_path).convert("RGB")
        label = torch.tensor(self.labels[idx])
        
        if self.transform:
            image = self.transform(image)
            
        return image, label, img_name # Trả về img_name để debug/visualize

In [29]:
# --- 5. TRANSFORMS (with Random Erasing) ---
def get_transforms():
    train_transform = transforms.Compose([
        transforms.Resize(IMAGE_SIZE),
        transforms.CenterCrop(IMAGE_SIZE),
        transforms.RandomHorizontalFlip(),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
    ])

    val_transform = transforms.Compose([
        transforms.Resize(IMAGE_SIZE),
        transforms.CenterCrop(IMAGE_SIZE),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
    ])
    return train_transform, val_transform

# --- 6. XAI CLASSES (GradCAM++ & IG) ---
class GradCAMPlusPlus:
    def __init__(self, model, target_layer):
        self.model = model
        self.target_layer = target_layer
        self.gradients = None
        self.activations = None
        self.handles = []
        
    def save_activation(self, module, input, output):
        self.activations = output.detach()
        
    def save_gradient(self, module, grad_input, grad_output):
        self.gradients = grad_output[0].detach()
        
    def register_hooks(self):
        self.target_layer._backward_hooks.clear()
        self.target_layer._forward_hooks.clear()
        self.target_layer._is_full_backward_hook = None
        
        self.handles.append(self.target_layer.register_forward_hook(self.save_activation))
        self.handles.append(self.target_layer.register_full_backward_hook(self.save_gradient))
        
    def remove_hooks(self):
        for handle in self.handles:
            handle.remove()
        self.handles = []
        
    def generate_cam(self, input_tensor, attribute_idx, target_class=1):
        input_tensor = input_tensor.clone().detach().requires_grad_(True)
        self.model.eval()
        output = self.model(input_tensor)
        
        # Sử dụng log probability thay vì raw logit
        if target_class == 1:
            # P(positive) = sigmoid(logit)
            # log P(positive) = -log(1 + exp(-logit))
            score = torch.sigmoid(output[0, attribute_idx])
        else:
            # P(negative) = 1 - sigmoid(logit) = sigmoid(-logit)
            # Tính gradient w.r.t probability của negative class
            score = 1 - torch.sigmoid(output[0, attribute_idx])
        
        # Dùng log để gradient scaling tốt hơn
        score = torch.log(score + 1e-8)
        
        self.model.zero_grad()
        score.backward(retain_graph=True)
        
        grads = self.gradients
        activations = self.activations
        
        grad_2 = grads.pow(2)
        grad_3 = grad_2 * grads
        eps = 1e-8
        spatial_sum = (activations * grad_3).sum(dim=(2, 3), keepdim=True)
        denom = 2 * grad_2 + spatial_sum
        denom = torch.clamp(denom, min=eps)
        alphas = grad_2 / denom
        
        weights = (alphas * torch.relu(grads)).sum(dim=(2, 3), keepdim=True)
        cam = torch.sum(weights * activations, dim=1, keepdim=True)
        cam = torch.relu(cam)
        cam = cam - cam.min()
        cam = cam / (cam.max() + eps)
        
        return cam.squeeze().cpu().detach().numpy()

def integrated_gradients(model, input_image, attribute_idx, target_class=1, baseline=None, steps=50, device=None):
    model.eval()
    if device is None:
        device = next(model.parameters()).device
    input_image = input_image.to(device)
    if baseline is None:
        baseline = torch.zeros_like(input_image).to(device)
    else:
        baseline = baseline.to(device)
        
    diff = input_image - baseline
    batch_size = 10
    all_grads = []
    
    for start in range(0, steps + 1, batch_size):
        end = min(start + batch_size, steps + 1)
        interpolated_images = []
        for k in range(start, end):
            alpha = k / steps
            interpolated_image = baseline + alpha * diff
            interpolated_images.append(interpolated_image)
        
        interpolated_inputs = torch.cat(interpolated_images, dim=0)
        interpolated_inputs.requires_grad_(True)
        outputs = model(interpolated_inputs)
        
        score = outputs[:, attribute_idx].sum()
        if target_class == 0:
            score = -score
            
        model.zero_grad()
        score.backward()
        grads = interpolated_inputs.grad.detach().clone()
        all_grads.append(grads)
        interpolated_inputs.grad = None
        
    all_grads = torch.cat(all_grads, dim=0)
    avg_grads = torch.mean(all_grads, dim=0, keepdim=True)
    ig_attributions = diff * avg_grads
    
    # Convert to heatmap (sum over channels and relu)
    attr_map = torch.sum(torch.abs(ig_attributions), dim=1, keepdim=True)
    attr_map = attr_map - attr_map.min()
    attr_map = attr_map / (attr_map.max() + 1e-8)
    
    return attr_map.squeeze().cpu().numpy()

def visualize_predictions(model, dataset, device, epoch):
    """
    Lấy 1 ảnh random, chạy GradCAM++ và IG cho 3 attributes quan trọng (ngẫu nhiên)
    Vẽ: Ảnh gốc | GradCAM++ Pos | GradCAM++ Neg | IG Pos | IG Neg
    Heatmap: Xanh (ít liên quan) -> Đỏ (liên quan mật thiết)
    """
    idx = np.random.randint(len(dataset))
    img_tensor, label_tensor, _ = dataset[idx]
    input_tensor = img_tensor.unsqueeze(0).to(device)
    
    # Denormalize image for display
    inv_normalize = transforms.Normalize(
        mean=[-0.485/0.229, -0.456/0.224, -0.406/0.225],
        std=[1/0.229, 1/0.224, 1/0.225]
    )
    disp_img = inv_normalize(img_tensor).permute(1, 2, 0).numpy()
    disp_img = np.clip(disp_img, 0, 1)

    # Get Prediction
    model.eval()
    with torch.no_grad():
        logits = model(input_tensor)
        probs = torch.sigmoid(logits).squeeze().cpu().numpy()
    
    # Setup GradCAM
    target_layer = model.layer4
    grad_cam = GradCAMPlusPlus(model, target_layer)
    grad_cam.register_hooks()

    # Chọn 3 attributes để plot
    true_indices = np.where(label_tensor.numpy() == 1)[0]
    false_indices = np.where(label_tensor.numpy() == 0)[0]
    
    plot_indices = []
    if len(true_indices) > 0: plot_indices.append(np.random.choice(true_indices))
    if len(false_indices) > 0: plot_indices.append(np.random.choice(false_indices))
    while len(plot_indices) < 3:
        rand_idx = np.random.randint(NUM_CLASSES)
        if rand_idx not in plot_indices:
            plot_indices.append(rand_idx)
            
    fig, axes = plt.subplots(len(plot_indices), 5, figsize=(20, 5 * len(plot_indices)))
    plt.suptitle(f"XAI Visualization - Epoch {epoch}", fontsize=16)
    
    for i, attr_idx in enumerate(plot_indices):
        attr_name = list(ATTR_MAP.values())[attr_idx]
        gt = int(label_tensor[attr_idx].item())
        pred_prob = probs[attr_idx]
        pred_label = 1 if pred_prob > 0.5 else 0
        
        title_str = f"{attr_name}\nGT: {gt} | Pred: {pred_label} ({pred_prob:.2f})"
        
        # Generate heatmaps
        cam_pos = grad_cam.generate_cam(input_tensor, attr_idx, target_class=1)
        cam_neg = grad_cam.generate_cam(input_tensor, attr_idx, target_class=0)
        ig_pos = integrated_gradients(model, input_tensor, attr_idx, target_class=1, steps=30, device=device)
        ig_neg = integrated_gradients(model, input_tensor, attr_idx, target_class=0, steps=30, device=device)

        # Resize heatmaps to match image size
        
        cam_pos_resized = resize(cam_pos, (IMAGE_SIZE, IMAGE_SIZE), mode='reflect', anti_aliasing=True)
        cam_neg_resized = resize(cam_neg, (IMAGE_SIZE, IMAGE_SIZE), mode='reflect', anti_aliasing=True)
        ig_pos_resized = resize(ig_pos, (IMAGE_SIZE, IMAGE_SIZE), mode='reflect', anti_aliasing=True)
        ig_neg_resized = resize(ig_neg, (IMAGE_SIZE, IMAGE_SIZE), mode='reflect', anti_aliasing=True)

        ax_row = axes[i] if len(plot_indices) > 1 else axes
        
        # 1. Original Image
        ax_row[0].imshow(disp_img)
        ax_row[0].set_title(title_str, fontsize=10)
        ax_row[0].axis('off')
        
        # 2. GradCAM++ Positive (Xanh -> Đỏ)
        ax_row[1].imshow(disp_img)
        heatmap = ax_row[1].imshow(cam_pos_resized, cmap='jet', alpha=0.6, vmin=0, vmax=1)
        ax_row[1].set_title("GradCAM++ Positive\n(Blue: Low | Red: High)", fontsize=10)
        ax_row[1].axis('off')

        # 3. GradCAM++ Negative (Xanh -> Đỏ)
        ax_row[2].imshow(disp_img)
        ax_row[2].imshow(cam_neg_resized, cmap='jet', alpha=0.6, vmin=0, vmax=1)
        ax_row[2].set_title("GradCAM++ Negative\n(Blue: Low | Red: High)", fontsize=10)
        ax_row[2].axis('off')
        
        # 4. IG Positive (Xanh -> Đỏ)
        ax_row[3].imshow(disp_img)
        ax_row[3].imshow(ig_pos_resized, cmap='jet', alpha=0.6, vmin=0, vmax=1)
        ax_row[3].set_title("IG Positive\n(Blue: Low | Red: High)", fontsize=10)
        ax_row[3].axis('off')

        # 5. IG Negative (Xanh -> Đỏ)
        ax_row[4].imshow(disp_img)
        ax_row[4].imshow(ig_neg_resized, cmap='jet', alpha=0.6, vmin=0, vmax=1)
        ax_row[4].set_title("IG Negative\n(Blue: Low | Red: High)", fontsize=10)
        ax_row[4].axis('off')

    # Add colorbar để hiểu scale
    fig.colorbar(heatmap, ax=axes.ravel().tolist(), label='Attribution Intensity', shrink=0.6)
    
    plt.tight_layout()
    plt.savefig(os.path.join(PATH_CHECKPOINT, f"viz_epoch_{epoch}.png"), dpi=150, bbox_inches='tight')
    plt.close()
    grad_cam.remove_hooks()
    print(f"Saved visualization to {os.path.join(PATH_CHECKPOINT, f'viz_epoch_{epoch}.png')}")

In [17]:
def visualize_specific_attribute(model, dataset, device, attr_name, num_samples=5):
    """
    Visualize samples với ground truth = 1 cho một attribute cụ thể
    """
    attr_idx = TARGET_ATTRS.index(attr_name)
    
    # Tìm các sample có attribute = 1
    positive_indices = []
    for i in range(len(dataset)):
        _, label, _ = dataset[i]
        if label[attr_idx] == 1:
            positive_indices.append(i)
        if len(positive_indices) >= num_samples:
            break
    
    print(f"Found {len(positive_indices)} positive samples for {attr_name}")
    
    for idx in positive_indices:
        img_tensor, label_tensor, img_name = dataset[idx]
        input_tensor = img_tensor.unsqueeze(0).to(device)
        
        # Setup GradCAM
        target_layer = model.cbam4
        grad_cam = GradCAMPlusPlus(model, target_layer)
        grad_cam.register_hooks()
        
        cam = grad_cam.generate_cam(input_tensor, attr_idx, target_class=1)
        
        # Denormalize
        inv_normalize = transforms.Normalize(
            mean=[-0.485/0.229, -0.456/0.224, -0.406/0.225],
            std=[1/0.229, 1/0.224, 1/0.225]
        )
        disp_img = inv_normalize(img_tensor).permute(1, 2, 0).numpy()
        disp_img = np.clip(disp_img, 0, 1)
        
        # Plot
        fig, axes = plt.subplots(1, 2, figsize=(10, 5))
        axes[0].imshow(disp_img)
        axes[0].set_title(f"{img_name}\nGT: {int(label_tensor[attr_idx])}")
        axes[0].axis('off')
        
        cam_resized = resize(cam, (IMAGE_SIZE, IMAGE_SIZE), mode='reflect', anti_aliasing=True)
        axes[1].imshow(disp_img)
        axes[1].imshow(cam_resized, cmap='jet', alpha=0.6)
        axes[1].set_title(f"GradCAM++ for {ATTR_MAP[attr_name]}")
        axes[1].axis('off')
        
        plt.tight_layout()
        plt.savefig(os.path.join(PATH_CHECKPOINT, f"debug_{attr_name}_{idx}.png"))
        plt.close()
        
        grad_cam.remove_hooks()

In [5]:
# --- INIT DATA & MODEL ---
train_tf, val_tf = get_transforms()

full_dataset = CelebADataset(CSV_PATH, IMAGE_PATH, TARGET_ATTRS, transform=train_tf) # Temp init to get length
# Split 9/1
train_size = int(0.9 * len(full_dataset))
val_size = len(full_dataset) - train_size

# Re-init datasets correctly to apply different transforms
train_dataset = CelebADataset(CSV_PATH, IMAGE_PATH, TARGET_ATTRS, transform=train_tf)
val_dataset = CelebADataset(CSV_PATH, IMAGE_PATH, TARGET_ATTRS, transform=val_tf)

# Using Subset to split indices but keep transforms
indices = torch.randperm(len(full_dataset)).tolist()
train_dataset = torch.utils.data.Subset(train_dataset, indices[:train_size])
val_dataset = torch.utils.data.Subset(val_dataset, indices[train_size:])

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=8, pin_memory=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=8, pin_memory=True) # Batch size can stay high if validation logic is efficient

print(f"Train samples: {len(train_dataset)}, Val samples: {len(val_dataset)}")

model = ResNet18_CBAM(num_classes=NUM_CLASSES).to(DEVICE)
print("Calculating positive weights for loss function...")

train_df = pd.read_csv(CSV_PATH).iloc[indices[:train_size]]
train_labels = train_df[TARGET_ATTRS].values
train_labels[train_labels == -1] = 0
train_labels = train_labels.astype(np.float32)
num_pos = np.sum(train_labels, axis=0)
num_neg = len(train_df) - num_pos
# Thêm epsilon để tránh chia cho 0
pos_weights_tensor = torch.tensor(num_neg / (num_pos + 1e-5), dtype=torch.float32).to(DEVICE)

criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weights_tensor)
# criterion = AsymmetricLossOptimized(gamma_neg=4, gamma_pos=1, clip=0.05).to(DEVICE)
optimizer = optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=1e-4)
scheduler = optim.lr_scheduler.OneCycleLR(optimizer, max_lr=LEARNING_RATE, steps_per_epoch=len(train_loader), epochs=NUM_EPOCHS)

Train samples: 182339, Val samples: 20260
Calculating positive weights for loss function...


In [ ]:
# --- TRAIN LOOP (GLOBAL SCOPE) ---
best_val_acc = 0.0

for epoch in range(1, NUM_EPOCHS + 1):
    # --- TRAINING ---
    model.train()
    train_loss = 0.0
    loop = tqdm(train_loader, desc=f"Epoch {epoch}/{NUM_EPOCHS} [Train]")
    
    for imgs, labels, _ in loop:
        imgs = imgs.to(DEVICE)
        labels = labels.to(DEVICE)
        
        outputs = model(imgs)
        loss = criterion(outputs, labels)
        
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        scheduler.step()
        
        train_loss += loss.item()
        loop.set_postfix(loss=loss.item())
        
    avg_train_loss = train_loss / len(train_loader)
    
    # --- VALIDATION (SAFE MEMORY) ---
    model.eval()
    val_loss = 0.0
    
    # Accumulate metrics per attribute
    # Use CPU tensors to store counts to avoid VRAM overflow
    correct_counts = torch.zeros(NUM_CLASSES)
    total_counts = torch.zeros(NUM_CLASSES)
    
    # Debug: store sum of probs to see average prediction confidence
    prob_sums = torch.zeros(NUM_CLASSES)
    
    val_loop = tqdm(val_loader, desc=f"Epoch {epoch}/{NUM_EPOCHS} [Val]")
    
    with torch.no_grad():
        for imgs, labels, _ in val_loop:
            imgs = imgs.to(DEVICE)
            labels = labels.to(DEVICE)
            
            outputs = model(imgs)
            loss = criterion(outputs, labels)
            val_loss += loss.item()
            
            # Calculate metrics immediately and detach to CPU
            probs = torch.sigmoid(outputs)
            preds = (probs > 0.5).float()
            
            # Move to CPU for accumulation
            labels_cpu = labels.cpu()
            preds_cpu = preds.cpu()
            probs_cpu = probs.cpu()
            
            # Update counts
            correct_counts += (preds_cpu == labels_cpu).sum(dim=0).float()
            total_counts += labels_cpu.size(0)
            prob_sums += probs_cpu.sum(dim=0)
            
            # Don't store 'outputs' or 'imgs' list!
            
    avg_val_loss = val_loss / len(val_loader)
    attr_accuracies = (correct_counts / total_counts) * 100
    avg_probs = prob_sums / total_counts
    mean_acc = attr_accuracies.mean().item()
    
    print(f"\n--- Epoch {epoch} Results ---")
    print(f"Train Loss: {avg_train_loss:.4f} | Val Loss: {avg_val_loss:.4f} | Mean Val Acc: {mean_acc:.2f}%")
    print(f"{'Attribute':<20} | {'Acc (%)':<10} | {'Avg Prob':<10}")
    print("-" * 45)
    for i, attr in enumerate(TARGET_ATTRS):
        print(f"{ATTR_MAP[attr]:<20} | {attr_accuracies[i]:.2f}%     | {avg_probs[i]:.4f}")
    
    # --- CHECKPOINT ---
    if epoch % 5 == 0:
        ckpt_name = f"mobilenetv3_epoch_{epoch}.pth"
        save_path = os.path.join(PATH_CHECKPOINT, ckpt_name)
        torch.save({
            'epoch': epoch,
            'model_state_dict': model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'loss': avg_val_loss,
        }, save_path)
        print(f"Saved Checkpoint: {save_path}")
        
        # --- VISUALIZATION (Every 5 epochs) ---
        print("Generating XAI visualization...")
        # Use the raw val_dataset (not Subset) to easily get item, 
        # but we need to map indices if using Subset. 
        # Simplest way: just take a batch from val_loader and pick one image.
    visualize_predictions(model, val_dataset, DEVICE, epoch)
    

print("Training Complete.")

Epoch 1/30 [Val]: 100%|██████████| 159/159 [00:20<00:00,  7.90it/s]



--- Epoch 1 Results ---
Train Loss: 0.1980 | Val Loss: 0.1122 | Mean Val Acc: 93.91%
Attribute            | Acc (%)    | Avg Prob  
---------------------------------------------
Smiling              | 93.09%     | 0.4702
Eyeglasses           | 99.47%     | 0.0791
Gender (Male)        | 98.28%     | 0.4194
Open Mouth           | 94.00%     | 0.4964
No Beard             | 93.98%     | 0.7854
Age (Young)          | 84.65%     | 0.6534


/tmp/ipykernel_303751/3368189874.py:224: UserWarning: This figure includes Axes that are not compatible with tight_layout, so results might be incorrect.
  plt.tight_layout()


Saved visualization to /mnt/e/KLTN/GAN/classifier/checkpoints/mobilenetv3_classifier_bce_eras/viz_epoch_1.png


Epoch 2/30 [Val]: 100%|██████████| 159/159 [00:17<00:00,  8.89it/s]


...


--- Epoch 6 Results ---
Train Loss: 0.1166 | Val Loss: 0.1353 | Mean Val Acc: 94.48%
Attribute            | Acc (%)    | Avg Prob  
---------------------------------------------
Smiling              | 92.89%     | 0.4967
Eyeglasses           | 99.34%     | 0.0767
Gender (Male)        | 97.36%     | 0.3987
Open Mouth           | 93.50%     | 0.5092
No Beard             | 95.33%     | 0.8413
Age (Young)          | 88.45%     | 0.7841
Saved visualization to /mnt/e/KLTN/GAN/classifier/checkpoints/mobilenetv3_classifier_bce_eras/viz_epoch_6.png


Epoch 7/30 [Val]: 100%|██████████| 159/159 [00:19<00:00,  8.35it/s]



--- Epoch 7 Results ---
Train Loss: 0.1135 | Val Loss: 0.1124 | Mean Val Acc: 94.29%
Attribute            | Acc (%)    | Avg Prob  
---------------------------------------------
Smiling              | 93.12%     | 0.4925
Eyeglasses           | 99.50%     | 0.0721
Gender (Male)        | 98.34%     | 0.4158
Open Mouth           | 94.13%     | 0.4934
No Beard             | 95.63%     | 0.8247
Age (Young)          | 84.99%     | 0.6518
Saved visualization to /mnt/e/KLTN/GAN/classifier/checkpoints/mobilenetv3_classifier_bce_eras/viz_epoch_7.png


Epoch 8/30 [Train]:  30%|███       | 434/1425 [01:25<03:14,  5.09it/s, loss=0.0918]


KeyboardInterrupt: 

In [27]:
PATH_CHECKPOINT = r"/mnt/e/KLTN/GAN/classifier/checkpoints/mobilenetv3_classifier"
model = ResNet18_CBAM(num_classes=NUM_CLASSES).to(DEVICE)
checkpoint = torch.load(os.path.join(PATH_CHECKPOINT, "mobilenetv3_epoch_5.pth"))
model.load_state_dict(checkpoint['model_state_dict'])

<All keys matched successfully>

In [31]:

visualize_specific_attribute(model, val_dataset, DEVICE, 'Smiling', num_samples=10)

Found 10 positive samples for Smiling


In [30]:
def visualize_specific_attribute_negative(model, dataset, device, attr_name, num_samples=5):
    save_path_dir = PATH_CHECKPOINT+"/negative_samples"
    if not os.path.exists(save_path_dir):
        os.makedirs(save_path_dir)
    """
    Visualize samples với ground truth = 0 cho một attribute cụ thể
    """
    attr_idx = TARGET_ATTRS.index(attr_name)
    
    # Tìm các sample có attribute = 0 (negative)
    negative_indices = []
    for i in range(len(dataset)):
        _, label, _ = dataset[i]
        if label[attr_idx] == 0:
            negative_indices.append(i)
        if len(negative_indices) >= num_samples:
            break
    
    print(f"Found {len(negative_indices)} negative samples for {attr_name}")
    
    for idx in negative_indices:
        img_tensor, label_tensor, img_name = dataset[idx]
        input_tensor = img_tensor.unsqueeze(0).to(device)
        
        # Setup GradCAM
        target_layer = model.cbam4
        grad_cam = GradCAMPlusPlus(model, target_layer)
        grad_cam.register_hooks()
        
        # Generate CAM for negative class (target_class=0)
        cam = grad_cam.generate_cam(input_tensor, attr_idx, target_class=0)
        
        # Denormalize
        inv_normalize = transforms.Normalize(
            mean=[-0.485/0.229, -0.456/0.224, -0.406/0.225],
            std=[1/0.229, 1/0.224, 1/0.225]
        )
        disp_img = inv_normalize(img_tensor).permute(1, 2, 0).numpy()
        disp_img = np.clip(disp_img, 0, 1)
        
        # Get prediction
        model.eval()
        with torch.no_grad():
            logits = model(input_tensor)
            prob = torch.sigmoid(logits[0, attr_idx]).item()
        
        # Plot
        fig, axes = plt.subplots(1, 2, figsize=(10, 5))
        axes[0].imshow(disp_img)
        axes[0].set_title(f"{img_name}\nGT: {int(label_tensor[attr_idx])} | Pred: {prob:.3f}")
        axes[0].axis('off')
        
        cam_resized = resize(cam, (IMAGE_SIZE, IMAGE_SIZE), mode='reflect', anti_aliasing=True)
        axes[1].imshow(disp_img)
        axes[1].imshow(cam_resized, cmap='jet', alpha=0.6)
        axes[1].set_title(f"GradCAM++ for NOT {ATTR_MAP[attr_name]}")
        axes[1].axis('off')
        
        plt.tight_layout()
        plt.savefig(os.path.join(save_path_dir, f"debug_{attr_name}_negative_{idx}.png"))
        plt.close()
        
        grad_cam.remove_hooks()

# Chạy visualize negative samples
visualize_specific_attribute_negative(model, val_dataset, DEVICE, 'Eyeglasses', num_samples=50)
visualize_specific_attribute_negative(model, val_dataset, DEVICE, 'No_Beard', num_samples=50)
visualize_specific_attribute_negative(model, val_dataset, DEVICE, 'Mouth_Slightly_Open', num_samples=50)
visualize_specific_attribute_negative(model, val_dataset, DEVICE, 'Young', num_samples=50)
visualize_specific_attribute_negative(model, val_dataset, DEVICE, 'Smiling', num_samples=50)
visualize_specific_attribute_negative(model, val_dataset, DEVICE, 'Male', num_samples=50)

Found 50 negative samples for Eyeglasses
Found 50 negative samples for No_Beard
Found 50 negative samples for Mouth_Slightly_Open
Found 50 negative samples for Young
Found 50 negative samples for Smiling
Found 50 negative samples for Male
